In [0]:
%run ./_common

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

log = get_logger("opspulse.gold")

# event_type -> (category, weight). Weights are a design choice, documented in the README.
DIM_ROWS = [
    ("WatchEvent", "attention", 3),
    ("ForkEvent", "attention", 4),
    ("PullRequestEvent", "collaboration", 3),
    ("PullRequestReviewEvent", "collaboration", 2),
    ("PullRequestReviewCommentEvent", "collaboration", 1),
    ("IssuesEvent", "collaboration", 2),
    ("IssueCommentEvent", "collaboration", 1),
    ("DiscussionEvent", "collaboration", 1),
    ("ReleaseEvent", "release", 3),
    ("PushEvent", "code", 1),
    ("CreateEvent", "code", 1),
    ("DeleteEvent", "code", 0),
]


def build_dim_event_type():
    """Small dimension table; overwritten on every run (idempotent)."""
    df = spark.createDataFrame(DIM_ROWS, ["event_type", "category", "weight"])
    df.write.format("delta").mode("overwrite").saveAsTable(DIM_EVENT_TYPE)
    log.info("dim_event_type rows: %d", df.count())


build_dim_event_type()

In [0]:
def enrich_events(silver_df, dim_df):
    """Broadcast-join the small dimension; unknown event types fall back to defaults."""
    return (
        silver_df.join(F.broadcast(dim_df), on="event_type", how="left")
        .withColumn("category", F.coalesce("category", F.lit("other")))
        .withColumn("weight", F.coalesce("weight", F.lit(1)))
    )


def build_hourly_activity(enriched):
    return enriched.groupBy("event_date", "event_hour").agg(
        F.count("*").alias("events"),
        F.countDistinct("repo_id").alias("active_repos"),
        F.countDistinct("actor_login").alias("active_actors"),
        F.sum((F.col("event_type") == "PushEvent").cast("int")).alias("push_events"),
    )


def build_event_type_mix(enriched):
    counts = enriched.groupBy("event_date", "event_hour", "event_type", "category").agg(
        F.count("*").alias("events")
    )
    per_hour = Window.partitionBy("event_date", "event_hour")
    return counts.withColumn(
        "share_pct", F.round(100 * F.col("events") / F.sum("events").over(per_hour), 2)
    )


MIN_UNIQUE_ACTORS = 3  # a repo needs several distinct users to count as "trending"


def build_trending_repos(enriched, top_n: int = 100):
    daily = (
        enriched.groupBy("event_date", "repo_id").agg(
            F.max("repo_name").alias("repo_name"),  # a repo can be renamed; keep one name
            F.sum("weight").alias("trend_score"),
            F.count("*").alias("events"),
            F.countDistinct("actor_login").alias("unique_actors"),
            F.sum((F.col("event_type") == "WatchEvent").cast("int")).alias("stars"),
            F.sum((F.col("event_type") == "ForkEvent").cast("int")).alias("forks"),
        )
        .filter(F.col("unique_actors") >= MIN_UNIQUE_ACTORS)
    )
    w = Window.partitionBy("event_date").orderBy(
        F.desc("trend_score"), F.desc("unique_actors"), F.asc("repo_id")
    )
    return daily.withColumn("rank", F.row_number().over(w)).filter(F.col("rank") <= top_n)


def build_top_contributors(enriched, top_n: int = 100):
    per_actor = enriched.groupBy("event_date", "actor_login").agg(
        F.count("*").alias("events"),
        F.countDistinct("repo_id").alias("repos_touched"),
        F.sum("weight").alias("contrib_score"),
    )
    w = Window.partitionBy("event_date").orderBy(F.desc("contrib_score"), F.asc("actor_login"))
    return per_actor.withColumn("rank", F.row_number().over(w)).filter(F.col("rank") <= top_n)

In [0]:
def dates_to_process() -> list:
    """Dates whose Silver data is newer than what Gold last computed (or never computed)."""
    silver_latest = (
        spark.table(SILVER_TABLE)
        .groupBy("event_date")
        .agg(F.max("bronze_ingest_ts").alias("latest_ingest"))
    )
    if not spark.catalog.tableExists(GOLD_HOURLY):
        return [r["event_date"] for r in silver_latest.collect()]

    gold_done = (
        spark.table(GOLD_HOURLY)
        .groupBy("event_date")
        .agg(F.max("computed_at").alias("computed_at"))
    )
    pending = silver_latest.join(gold_done, "event_date", "left").filter(
        F.col("computed_at").isNull() | (F.col("latest_ingest") > F.col("computed_at"))
    )
    return [r["event_date"] for r in pending.collect()]


def write_partitioned(df, table: str, partition_cols: list, dates: list) -> None:
    """First run creates the partitioned table; later runs overwrite only the given dates."""
    if spark.catalog.tableExists(table):
        date_list = ",".join(f"'{d}'" for d in dates)
        (
            df.write.format("delta").mode("overwrite")
            .option("replaceWhere", f"event_date IN ({date_list})")
            .saveAsTable(table)
        )
    else:
        (
            df.write.format("delta").mode("overwrite")
            .partitionBy(*partition_cols)
            .saveAsTable(table)
        )


def run_gold() -> None:
    dates = dates_to_process()
    if not dates:
        log.info("gold is up to date, nothing to do")
        return
    log.info("processing dates: %s", [str(d) for d in dates])

    silver = spark.table(SILVER_TABLE).filter(F.col("event_date").isin(dates))
    enriched = enrich_events(silver, spark.table(DIM_EVENT_TYPE))

    # hourly_activity is written LAST: it acts as the commit marker for dates_to_process()
    outputs = [
        (GOLD_TRENDING, build_trending_repos(enriched), ["event_date"]),
        (GOLD_CONTRIBUTORS, build_top_contributors(enriched), ["event_date"]),
        (GOLD_TYPE_MIX, build_event_type_mix(enriched), ["event_date", "event_hour"]),
        (GOLD_HOURLY, build_hourly_activity(enriched), ["event_date", "event_hour"]),
    ]
    try:
        for table, df, parts in outputs:
            write_partitioned(df.withColumn("computed_at", F.current_timestamp()), table, parts, dates)
            log.info("written: %s", table)
    except Exception as e:
        log.error("gold run failed: %s", e)
        raise


run_gold()

In [0]:
for t in [GOLD_TRENDING, GOLD_CONTRIBUTORS, GOLD_TYPE_MIX, GOLD_HOURLY]:
    print(t, "->", spark.table(t).count(), "rows")

# Join sanity: how many Silver events fell back to the default category?
enriched = enrich_events(spark.table(SILVER_TABLE), spark.table(DIM_EVENT_TYPE))
print("rows with fallback category 'other':", enriched.filter(F.col("category") == "other").count())

display(spark.table(GOLD_TRENDING).orderBy("event_date", "rank").limit(10))
display(spark.table(GOLD_HOURLY).orderBy("event_date", "event_hour"))
display(spark.table(GOLD_TYPE_MIX).filter(F.col("event_hour") == 12).orderBy(F.desc("events")))

# Partition layout (SQL only for this quick sanity check)
display(spark.sql(f"DESCRIBE DETAIL {GOLD_HOURLY}").select("partitionColumns", "numFiles", "sizeInBytes"))